# Imports and paths START HERE

In [1]:
from pathlib import Path
import os
import torch
import gymnasium as gym
import ale_py

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

# Register Atari/ALE environments like ALE/Pong-v5
gym.register_envs(ale_py)

# Figure out project root whether notebook runs from /code or project root
cwd = Path.cwd()

if (cwd / "models").exists() and (cwd / "videos").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "models").exists() and (cwd.parent / "videos").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

CODE_DIR = PROJECT_ROOT / "code"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
VIDEO_DIR = PROJECT_ROOT / "videos"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
VIDEO_DIR.mkdir(exist_ok=True)

ENV_ID = "ALE/Pong-v5"
SEED = 0

# DQN commonly uses one environment because it learns from a replay buffer.
N_ENVS = 1

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)
print("Video directory:", VIDEO_DIR)
print("Environment:", ENV_ID)
print("Number of environments:", N_ENVS)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Current working directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\code
Project root: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3
Model directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models
Results directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\results
Video directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos
Environment: ALE/Pong-v5
Number of environments: 1
Torch version: 2.11.0+cu128
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


# DQN Pong Training

This notebook trains a Deep Q-Network baseline on `ALE/Pong-v5` using Stable-Baselines3. DQN is the required baseline algorithm for the project. The notebook saves checkpoints at multiple training levels and includes a reload/resume section for continuing training from a saved checkpoint.

# Create DQN Pong environment

In [4]:
ENV_ID = "ALE/Pong-v5"
SEED = 0

# DQN commonly uses one environment because it learns from a replay buffer.
N_ENVS = 1

env = make_atari_env(
    ENV_ID,
    n_envs=N_ENVS,
    seed=SEED,
    wrapper_kwargs=dict(terminal_on_life_loss=False),
)

env = VecFrameStack(env, n_stack=4)

obs = env.reset()

print("Environment:", ENV_ID)
print("Observation shape:", obs.shape)
print("Number of environments:", N_ENVS)

Environment: ALE/Pong-v5
Observation shape: (1, 84, 84, 4)
Number of environments: 1


# Create DQN model

## Fresh training only

Run the next environment/model/training-loop cells only if starting a new DQN training run from scratch.

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DQN(
    "CnnPolicy",
    env,
    verbose=1,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=32,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=1_000,
    exploration_fraction=0.10,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,
    device=device,
)

print("DQN model created.")
print("Model device:", model.device)

Using cuda device
Wrapping the env in a VecTransposeImage.


c:\Users\dosan\miniconda3\envs\sb3-atari\lib\site-packages\stable_baselines3\common\buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 5.65GB > 4.52GB
  warnings.warn(


DQN model created.
Model device: cuda


# Small DQN checkpoint training loop

In [6]:
# Small test checkpoints
checkpoint_steps = [10_000, 20_000, 30_000]
# checkpoint_steps = [100_000, 250_000, 500_000]
# checkpoint_steps = [100_000, 250_000, 500_000, 1_000_000]

previous_steps = 0

for target_steps in checkpoint_steps:
    steps_to_train = target_steps - previous_steps
    
    print("=" * 60)
    print(f"Training DQN from {previous_steps:,} to {target_steps:,} timesteps")
    print(f"Training for {steps_to_train:,} new timesteps")
    print("=" * 60)
    
    model.learn(
        total_timesteps=steps_to_train,
        reset_num_timesteps=False
    )
    
    checkpoint_path = MODEL_DIR / f"dqn_pong_{target_steps}.zip"
    model.save(checkpoint_path)
    
    print(f"Saved checkpoint: {checkpoint_path}")
    
    previous_steps = target_steps

print("DQN small checkpoint test complete.")

Training DQN from 0 to 10,000 timesteps
Training for 10,000 new timesteps
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 880      |
|    ep_rew_mean      | -20.2    |
|    exploration_rate | 0.145    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 701      |
|    time_elapsed     | 1        |
|    total_timesteps  | 864      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 862      |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 698      |
|    time_elapsed     | 2        |
|    total_timesteps  | 1688     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 865      |
|    ep_rew_mean      | -20.7    |
|    exploration

# Close environment after training

In [7]:
env.close()
print("Training environment closed.")

Training environment closed.


# Continue training from a saved DQN checkpoint, IMPORTANT START HERE FIRST

If reopening this notebook later, run the START HERE setup cell first, then start from this section. Do not recreate a brand-new DQN model if the goal is to continue a previous run.

# Load DQN checkpoint

In [4]:
# Choose which DQN checkpoint to resume from
resume_from_steps = 2_000_000

# How much to train before saving each new checkpoint
extra_training_steps = 1_000_000

# How many times to repeat the extra training
num_resume_runs = 3

reload_path = MODEL_DIR / f"dqn_pong_{resume_from_steps}.zip"

if not reload_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {reload_path}")

print("Loading checkpoint:", reload_path)

# Recreate the same kind of environment used for training
resume_env = make_atari_env(
    ENV_ID,
    n_envs=N_ENVS,
    seed=SEED,
    wrapper_kwargs=dict(terminal_on_life_loss=False),
)

resume_env = VecFrameStack(resume_env, n_stack=4)

# Load selected checkpoint
loaded_model = DQN.load(
    reload_path,
    env=resume_env,
    device=device,
)

print("Loaded DQN model successfully.")
print("Loaded model device:", loaded_model.device)

Loading checkpoint: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models\dqn_pong_2000000.zip
Wrapping the env in a VecTransposeImage.
Loaded DQN model successfully.
Loaded model device: cuda


c:\Users\dosan\miniconda3\envs\sb3-atari\lib\site-packages\stable_baselines3\common\buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 5.65GB > 3.26GB
  warnings.warn(


# Resume DQN training from loaded checkpoint

In [5]:
current_steps = resume_from_steps

for run_index in range(1, num_resume_runs + 1):
    next_steps = current_steps + extra_training_steps

    print("=" * 60)
    print(f"Resume run {run_index}/{num_resume_runs}")
    print(f"Training from {current_steps:,} to {next_steps:,} timesteps")
    print(f"Training for {extra_training_steps:,} additional timesteps")
    print("=" * 60)

    loaded_model.learn(
        total_timesteps=extra_training_steps,
        reset_num_timesteps=False
    )

    checkpoint_path = MODEL_DIR / f"a2c_pong_{next_steps}.zip"
    loaded_model.save(checkpoint_path)

    print(f"Saved DQN checkpoint: {checkpoint_path}")

    current_steps = next_steps

resume_env.close()
print("Resume environment closed.")
print("Resume checkpoint loop complete.")

Resume run 1/3
Training from 2,000,000 to 3,000,000 timesteps
Training for 1,000,000 additional timesteps
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 5.03e+03 |
|    ep_rew_mean      | -3.17    |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 2472     |
|    fps              | 265      |
|    time_elapsed     | 17       |
|    total_timesteps  | 2004569  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00411  |
|    n_updates        | 498642   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 5.05e+03 |
|    ep_rew_mean      | -3.18    |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 2476     |
|    fps              | 261      |
|    time_elapsed     | 37       |
|    total_timesteps  | 2009893  |
| train/           